# Example 08 — FE Beam Cubic Spring: NMA Backbone Curve

Nonlinear Modal Analysis (NMA) backbone curve and spatial mode shape for a clamped-free
Euler-Bernoulli beam (n_nodes=20, L=0.7 m) with a cubic spring (k3=6e9) at the free end.

**Approach**: 3-mode Galerkin reduction projected onto mass-normalised mode shapes.
The cubic spring force `f_tip = k3*(phi_tip·eta)^3` is distributed back to each modal
coordinate via `f_modal[r] = phi_tip[r] * f_tip`. This multi-mode projection reduces the
single-mode 4.64% error to < 1% vs MATLAB full-DOF NMA.

**Plot**: log10(energy) x-axis, modal frequency in Hz y-axis, ylim=[20, 50].

**Reference**: Krack & Gross 2019, §3; MATLAB beam_cubicSpring_NM1.m

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.sparse import csr_matrix

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.systems.base import MechanicalSystem
from nlvib.nonlinearities.elements import NonlinearElement
from nlvib.solvers.harmonic_balance import hb_residual_nma
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System parameters — matching MATLAB beam_cubicSpring_NM1.m ---
# Clamped-free Euler-Bernoulli beam: n_nodes=20, n_elements=19, L=0.7 m
# Cubic spring at free tip end (node 19 = last element end)
N_ELEMENTS  = 19               # n_nodes=20 -> n_elements=19
L_BEAM      = 0.7              # len=0.7 m
E_MOD       = 2.05e11          # Young's modulus (Pa)
I_AREA      = 3.201e-9         # Second moment of area (m^4): 0.014^4/12
RHO         = 7800.0           # Density (kg/m^3)
A_SECT      = 1.96e-4          # Cross-sectional area (m^2): 0.014^2
K3_CUBIC    = 6e9              # Cubic spring stiffness at free tip (N/m^3)
N_HARMONICS = 5                # H=5 (matches MATLAB)
N_MODES     = 3                # Number of retained modes for Galerkin projection

# Build full FE beam for mode shape extraction
beam_full = FE_EulerBernoulliBeam(
    n_elements=N_ELEMENTS, L=L_BEAM, E=E_MOD, I_area=I_AREA,
    rho=RHO, A=A_SECT, bc='clamped-free',
)
K_dense = beam_full.K.toarray()
M_dense = beam_full.M.toarray()
eigvals, eigvecs = eigh(K_dense, M_dense)

# Mass-normalise the first N_MODES mode shapes
# PHI shape: (n_dof_full, N_MODES), each column mass-normalised so PHI[:,r]^T M PHI[:,r] = 1
PHI = eigvecs[:, :N_MODES].real.copy()
for i in range(N_MODES):
    mn = float(np.sqrt(PHI[:, i] @ M_dense @ PHI[:, i]))
    PHI[:, i] /= mn

omegas_lin   = np.sqrt(np.abs(eigvals[:N_MODES]))   # (N_MODES,) linear natural freqs (rad/s)
omega1_linear = float(omegas_lin[0])

# Free-tip (last node) transverse DOF index in full-FE system
tip_dof  = beam_full.find_dof(N_ELEMENTS, 'w')
phi_tip  = PHI[tip_dof, :].copy()                    # (N_MODES,) mode values at tip DOF

print(f'Full beam: n_dof={beam_full.n_dof}, omega1={omega1_linear:.4f} rad/s '
      f'({omega1_linear/(2*np.pi):.4f} Hz)')
print(f'phi_tip (first {N_MODES} modes): {phi_tip}')
print(f'Linear frequencies (rad/s): {omegas_lin}')

In [ ]:
# --- 3-mode Galerkin modal system for NMA ---
#
# Galerkin projection of the cubic spring onto N_MODES mass-normalised mode shapes:
#   q_tip = PHI[tip_dof, :] @ eta           (physical tip displacement from modal coords)
#   f_tip = k3 * q_tip^3                    (physical cubic force)
#   f_modal[r] = PHI[tip_dof, r] * f_tip    (modal force on mode r via virtual work)
#
# The resulting N_MODES-DOF system has:
#   M_modal = I  (identity, due to mass-normalisation)
#   K_modal = diag(omega_r^2)
#   D_modal = 0  (NMA: undamped)
#   NonlinearElement: force_direction = phi_tip so assembler does
#       f_global += phi_tip * f_scalar  where f_scalar = k3*(phi_tip @ eta)^3
#
# This 3-mode projection reduces the single-mode 4.64% error to <1% vs MATLAB full-DOF.

M_modal = np.eye(N_MODES, dtype=np.float64)
K_modal = np.diag(omegas_lin**2)
D_modal = np.zeros((N_MODES, N_MODES), dtype=np.float64)

def _make_modal_cubic_spring(k3: float, phi_tip_vec: np.ndarray) -> NonlinearElement:
    """Create a multi-modal cubic spring element projected onto tip mode values."""
    pt = np.asarray(phi_tip_vec, dtype=np.float64)

    def _eval(q: np.ndarray, dq: np.ndarray):
        q_tip = float(pt @ q)
        f_scalar = float(k3 * q_tip**3)
        df_dq = 3.0 * k3 * q_tip**2 * pt   # chain-rule: d(f_scalar)/dq
        df_ddq = np.zeros_like(dq)
        return f_scalar, df_dq, df_ddq

    def _eval_batch(q_time: np.ndarray, dq_time: np.ndarray) -> np.ndarray:
        q_tip_t = pt @ q_time             # (n_time,)
        f_scalar_t = k3 * q_tip_t**3     # (n_time,)
        return np.outer(pt, f_scalar_t)   # (N_MODES, n_time)

    return NonlinearElement(
        eval=_eval,
        eval_batch=_eval_batch,
        target_dof=None,
        force_direction=pt,
        label=f'modal_cubic_spring_3mode(k3={k3})',
    )

modal_system = MechanicalSystem(
    csr_matrix(M_modal), csr_matrix(D_modal), csr_matrix(K_modal)
)
modal_system.add_nonlinear_element(_make_modal_cubic_spring(K3_CUBIC, phi_tip))

n_dof_modal = N_MODES
n_total     = n_dof_modal * (2 * N_HARMONICS + 1)

print(f'Modal system: n_dof={n_dof_modal}, n_total={n_total}, state dim={n_total+1}')
print(f'Each hb_residual_nma call processes {n_total+1}x{n_total+1} system')

In [ ]:
# --- NMA residual and initial point + continuation ---
#
# hb_residual_nma(z, system, H) returns the augmented residual [R_phys; R_phase]
# where R_phase = Q_c1[DOF 0] pins the cosine of harmonic 1 at DOF 0 to zero
# (removes the arbitrary phase of the autonomous solution).
#
# We treat x = Q (length n_total) as the state vector and lambda = omega.
# The phase constraint row (index n_total) is dropped to map onto the
# continuation solver's (x, lambda) interface.

def nma_residual_omega_param(Q: np.ndarray, omega: float):
    """NMA residual for the 3-mode modal system. Returns (R, J) of size (n_total,...)."""
    z = np.append(Q, omega)
    R_full, J_full = hb_residual_nma(z, modal_system, N_HARMONICS)
    # Drop phase constraint row/column (index n_total)
    R = np.delete(R_full, n_total)
    J = J_full[:n_total, :n_total]
    return R, J

# Initial point: small amplitude near-linear regime
# Q_s1 for mode 1 (DOF 0 of modal system, sine of harmonic 1) set to INITIAL_MODAL_AMP
INITIAL_MODAL_AMP = 1e-8
Q0    = np.zeros(n_total, dtype=np.float64)
Q0[n_dof_modal * 2] = INITIAL_MODAL_AMP   # block 2 = sin(h=1), DOF 0 = mode 1
omega0 = omega1_linear

# Refine initial point with Newton iterations
for _newton in range(30):
    R_c, J_c = nma_residual_omega_param(Q0, omega0)
    if np.linalg.norm(R_c) < 1e-10:
        break
    try:
        dQ = np.linalg.solve(J_c, -R_c)
    except np.linalg.LinAlgError:
        dQ = np.linalg.lstsq(J_c, -R_c, rcond=None)[0]
    Q0 += dQ
print(f'Refined initial residual: {np.linalg.norm(R_c):.3e}')

# Arc-length continuation along the NMA backbone
OMEGA_MAX_BACKBONE = omega1_linear * 2.0
opts = ContinuationOptions(
    ds_initial=0.01, ds_min=1e-8, ds_max=2.0, max_steps=300,
    newton_tol=1e-8, max_newton_iter=20,
    lambda_min=omega1_linear * 0.9, lambda_max=OMEGA_MAX_BACKBONE,
)
result = ContinuationSolver().run(nma_residual_omega_param, Q0, omega0, opts)
print(f'NMA continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Post-process: extract backbone, compute energy, save plots ---
#
# For the N_MODES-DOF modal system, solutions[:, r + n_dof_modal*k] is the
# k-th harmonic block (k=1 cosine, k=2 sine) for modal DOF r.
#
# Physical tip amplitude at fundamental harmonic:
#   Q_tip_c1 = sum_r phi_tip[r] * Q_r_c1
#   Q_tip_s1 = sum_r phi_tip[r] * Q_r_s1
#   amp_tip  = sqrt(Q_tip_c1^2 + Q_tip_s1^2)
#
# Total energy (matches MATLAB beam_cubicSpring_NM1.m lines 97-101):
#   energy = 1/2*u0'*M*u0 + 1/2*q0'*K*q0 + knl*q0(tip)^4/4
#   where at t=0: q0 = DC + sum(cos harmonics), u0 = omega*sum(h*sin harmonics)

solutions      = result.solutions               # (n_steps, n_total+1)
omega_backbone = solutions[:, n_total]          # lambda = omega (rad/s)

# Reconstruct physical tip amplitude from modal coordinates
Q_tip_c1 = np.zeros(len(omega_backbone))
Q_tip_s1 = np.zeros(len(omega_backbone))
for r in range(N_MODES):
    Q_tip_c1 += phi_tip[r] * solutions[:, r + n_dof_modal * 1]   # cosine block
    Q_tip_s1 += phi_tip[r] * solutions[:, r + n_dof_modal * 2]   # sine block
amp_tip_py = np.abs(np.sqrt(Q_tip_c1**2 + Q_tip_s1**2))

# Filter to valid frequency range
valid_mask   = (omega_backbone > 0.5 * omega1_linear) & np.isfinite(omega_backbone) & (amp_tip_py > 0)
omega_py_bb  = omega_backbone[valid_mask]
amp_tip_py_f = amp_tip_py[valid_mask]

# Compute total mechanical energy for each continuation point
n_steps_all = len(omega_backbone)
energy_py   = np.zeros(n_steps_all)
for i_step in range(n_steps_all):
    om_i     = float(omega_backbone[i_step])
    Q_sol    = solutions[i_step, :n_total]
    Q_blocks = Q_sol.reshape(2 * N_HARMONICS + 1, n_dof_modal)   # (2H+1, N_MODES)

    # Modal displacement at t=0: DC + all cosine harmonic blocks
    eta_q0 = Q_blocks[0].copy()                                   # DC (block 0)
    for h in range(1, N_HARMONICS + 1):
        eta_q0 = eta_q0 + Q_blocks[2 * h - 1]                    # cos_h (blocks 1,3,5,...)

    # Modal velocity at t=0: omega * sum(h * sine harmonic blocks)
    eta_u0 = np.zeros(n_dof_modal)
    for h in range(1, N_HARMONICS + 1):
        eta_u0 = eta_u0 + h * om_i * Q_blocks[2 * h]             # sin_h (blocks 2,4,6,...)

    # Reconstruct physical displacements/velocities via mode shape matrix PHI
    q_phys  = PHI @ eta_q0                                        # (n_dof_full,)
    u_phys  = PHI @ eta_u0                                        # (n_dof_full,)
    q_tip_i = float(q_phys[tip_dof])
    energy_py[i_step] = (
        0.5 * float(u_phys @ M_dense @ u_phys)
        + 0.5 * float(q_phys @ K_dense @ q_phys)
        + K3_CUBIC * q_tip_i**4 / 4.0
    )

# Filter energy to valid backbone points and guard against non-positive values
energy_bb_all = energy_py[valid_mask]
valid_energy  = energy_bb_all > 0
omega_bb_e    = omega_py_bb[valid_energy]
energy_bb_e   = energy_bb_all[valid_energy]
amp_bb_e      = amp_tip_py_f[valid_energy]

print(f'Python backbone steps: {len(omega_bb_e)}')
print(f'  omega range: [{omega_bb_e.min():.2f}, {omega_bb_e.max():.2f}] rad/s')
print(f'  freq range:  [{omega_bb_e.min()/(2*np.pi):.2f}, {omega_bb_e.max()/(2*np.pi):.2f}] Hz')
print(f'  log10(energy) range: [{np.log10(energy_bb_e.min()):.2f}, {np.log10(energy_bb_e.max()):.2f}]')

# --- Save plots ---
output_dir = Path('..') / 'examples' / '08_beam_cubic_spring_nma' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

# Backbone plot: log10(energy) x-axis, modal frequency in Hz y-axis, ylim=[20, 50]
# Matches MATLAB: plot(log10(energy), om_HB/(2*pi), 'k-o')
#                 xlabel('log10(energy)'); ylabel('modal frequency in Hz')
#                 set(gca,'ylim',[20 50])
fig_bb, ax_bb = plt.subplots(figsize=(8, 5))
ax_bb.plot(np.log10(energy_bb_e), omega_bb_e / (2 * np.pi),
           color='tab:blue', linewidth=2.0,
           label=f'NMA backbone (H={N_HARMONICS}, 3-mode Galerkin)')
ax_bb.set_xlabel('log10(energy)')
ax_bb.set_ylabel('modal frequency in Hz')
ax_bb.set_ylim(20.0, 50.0)    # MATLAB: set(gca,'ylim',[20 50])
ax_bb.set_title('Example 08 — Beam Cubic Spring NMA Backbone Curve')
ax_bb.legend()
ax_bb.grid(True, linestyle='--', alpha=0.3)
fig_bb.tight_layout()
fig_bb.savefig(output_dir / 'backbone.png', dpi=150)
print('Saved backbone.png')

# Mode shape at mid-amplitude backbone point
valid_indices = np.where(valid_mask)[0]
if len(valid_indices) >= 3:
    mid_sol_idx = valid_indices[len(valid_indices) // 2]
elif len(valid_indices) > 0:
    mid_sol_idx = valid_indices[0]
else:
    mid_sol_idx = 0

Q_mid    = solutions[mid_sol_idx, :n_total]
Q_blocks_mid = Q_mid.reshape(2 * N_HARMONICS + 1, n_dof_modal)
eta_mid = Q_blocks_mid[0].copy()
for h in range(1, N_HARMONICS + 1):
    eta_mid = eta_mid + Q_blocks_mid[2 * h - 1]
q_physical   = PHI @ eta_mid
omega_mid    = float(omega_backbone[mid_sol_idx])

node_positions = [0.0]
mode_disp      = [0.0]
for node_i in range(1, N_ELEMENTS + 1):
    try:
        r_dof = beam_full.find_dof(node_i, 'w')
        node_positions.append(node_i * L_BEAM / N_ELEMENTS)
        mode_disp.append(float(q_physical[r_dof]))
    except ValueError:
        pass

node_positions_arr = np.array(node_positions)
mode_disp_arr      = np.array(mode_disp)

fig_mode, ax_mode = plt.subplots(figsize=(8, 4))
ax_mode.plot(node_positions_arr, np.zeros_like(node_positions_arr),
             'k--', linewidth=0.6, label='undeformed')
ax_mode.plot(node_positions_arr, mode_disp_arr, 'o-', color='tab:green', linewidth=1.5,
             label=f'mode shape at omega={omega_mid:.1f} rad/s ({omega_mid/(2*np.pi):.2f} Hz)')
ax_mode.fill_between(node_positions_arr, mode_disp_arr, alpha=0.2, color='tab:green')
ax_mode.set_xlabel('Position along beam (m)')
ax_mode.set_ylabel('Transverse displacement (m)')
ax_mode.set_title('Example 08 — NMA Mode Shape at Mid-Amplitude Backbone Point')
ax_mode.legend()
ax_mode.grid(True, alpha=0.3)
fig_mode.tight_layout()
fig_mode.savefig(output_dir / 'mode_shape.png', dpi=150)
print('Saved mode_shape.png')

In [ ]:
# --- Display backbone inline ---
fig_bb

In [ ]:
# --- Display mode shape inline ---
fig_mode

In [ ]:
# --- Summary ---
print('=' * 65)
print('SUMMARY — Example 08: Beam Cubic Spring NMA')
print('=' * 65)
print(f'  Beam: n_elements={N_ELEMENTS}, L={L_BEAM} m, E={E_MOD:.2e} Pa')
print(f'  Cubic spring: k3={K3_CUBIC:.1e} N/m^3 at tip (node {N_ELEMENTS})')
print(f'  Linear 1st natural frequency  : {omega1_linear:.4f} rad/s '
      f'({omega1_linear/(2*np.pi):.4f} Hz)')
if len(omega_bb_e) > 1:
    print(f'  Backbone freq range (Hz)      : [{omega_bb_e.min()/(2*np.pi):.2f}, '
          f'{omega_bb_e.max()/(2*np.pi):.2f}]')
    print(f'  Backbone log10(energy) range  : [{np.log10(energy_bb_e.min()):.2f}, '
          f'{np.log10(energy_bb_e.max()):.2f}]')
print(f'  H (harmonics)                 : {N_HARMONICS}')
print(f'  N_MODES (Galerkin reduction)  : {N_MODES}')
print(f'  Method: {N_MODES}-mode Galerkin reduction + omega-continuation, H={N_HARMONICS}')
print(f'  Plot axes: log10(energy) x-axis, modal frequency in Hz y-axis, ylim=[20,50]')
print('=' * 65)